[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/03-census-demographics.ipynb)

# Census Demographics

`get_census_data` retrieves demographic variables from the US Census Bureau's American Community Survey (ACS) 5-year estimates. It accepts multiple input formats and returns a structured `CensusDataResult`.

In this notebook you will learn how to:

1. Fetch census data for an isochrone area
2. Inspect the `CensusDataResult` object
3. Use friendly variable names
4. Fetch multiple variables at once
5. Use raw Census Bureau B-codes
6. Pass GEOIDs directly
7. Use a point location
8. Merge demographics with block geometries
9. Compute aggregate statistics

## Setup

In [ ]:
# Uncomment to install on Google Colab:
# !pip install socialmapper

from socialmapper import create_isochrone, get_census_blocks, get_census_data

## 1. Census Data from an Isochrone

The simplest workflow: create an isochrone, then pass it directly to `get_census_data`.

In [ ]:
iso = create_isochrone("Denver, CO", travel_time=15, travel_mode="drive")
census_result = get_census_data(iso, variables=["population"])

print(f"Location type: {census_result.location_type}")
print(f"Block groups: {len(census_result.data)}")

## 2. Inspect CensusDataResult

The result has three fields: `data`, `location_type`, and `query_info`.

In [ ]:
# query_info shows what was requested
for key, value in census_result.query_info.items():
    print(f"{key}: {value}")

In [ ]:
# data is a nested dict: {geoid: {variable: value}}
first_geoid = list(census_result.data.keys())[0]
print(f"GEOID: {first_geoid}")
print(f"Data:  {census_result.data[first_geoid]}")

## 3. Friendly Variable Names

SocialMapper maps common names to Census Bureau codes. Here are the available names:

| Friendly name | Census code | Description |
|---|---|---|
| `population` | B01003_001E | Total population |
| `median_income` | B19013_001E | Median household income |
| `median_age` | B01002_001E | Median age |
| `housing_units` | B25001_001E | Total housing units |
| `poverty` | B17001_002E | Population below poverty line |
| `occupied_housing` | B25003_001E | Occupied housing units |
| `owner_occupied` | B25003_002E | Owner-occupied housing |
| `renter_occupied` | B25003_003E | Renter-occupied housing |
| `white_population` | B02001_002E | White population |
| `black_population` | B02001_003E | Black population |
| `asian_population` | B02001_005E | Asian population |
| `hispanic_population` | B03002_012E | Hispanic population |
| `bachelors_degree` | B15003_022E | Pop. with bachelor's degree |
| `high_school` | B15003_017E | Pop. with high school diploma |
| `households_no_vehicle` | B08201_002E | Households with no vehicle |
| `median_home_value` | B25077_001E | Median home value |
| `median_rent` | B25064_001E | Median gross rent |

In [ ]:
result = get_census_data(iso, variables=["median_income"])
sample_geoid = list(result.data.keys())[0]
print(f"Median income for {sample_geoid}: ${result.data[sample_geoid].get('median_income', 'N/A'):,}")

## 4. Fetch Multiple Variables

In [ ]:
multi = get_census_data(iso, variables=["population", "median_income", "median_age", "housing_units", "poverty"])

print(f"Variables requested: {multi.query_info['variables']}")
print(f"Census codes used:  {multi.query_info['variable_codes']}")
print()

# Show first 3 block groups
for geoid in list(multi.data.keys())[:3]:
    print(f"{geoid}: {multi.data[geoid]}")

## 5. Raw Census B-Codes

You can also use raw ACS variable codes directly.

In [ ]:
raw = get_census_data(iso, variables=["B01003_001E", "B19013_001E"])
sample_geoid = list(raw.data.keys())[0]
print(f"Data for {sample_geoid}: {raw.data[sample_geoid]}")

## 6. GEOID-List Input

Pass a list of 12-digit GEOIDs directly if you already know which block groups you want.

In [ ]:
# Get some GEOIDs from our earlier isochrone
blocks = get_census_blocks(polygon=iso)
geoid_list = [b["geoid"] for b in blocks[:5]]
print(f"Querying GEOIDs: {geoid_list}")

geoid_result = get_census_data(geoid_list, variables=["population", "median_income"])
print(f"Location type: {geoid_result.location_type}")
for geoid, data in geoid_result.data.items():
    print(f"  {geoid}: {data}")

## 7. Point Location Input

Pass a `(latitude, longitude)` tuple to get data for the single block group at that point.

In [ ]:
# Colorado State Capitol
point_result = get_census_data((39.7392, -104.9903), variables=["population", "median_income"])
print(f"Location type: {point_result.location_type}")
print(f"Block groups returned: {len(point_result.data)}")
for geoid, data in point_result.data.items():
    print(f"  {geoid}: {data}")

## 8. Merge Demographics with Block Geometries

Combine census data with block group geometries for mapping (covered fully in Notebook 04).

In [ ]:
blocks = get_census_blocks(polygon=iso)
census = get_census_data(iso, variables=["population", "median_income"])

# Merge by GEOID
merged = []
for block in blocks:
    geoid = block["geoid"]
    if geoid in census.data:
        entry = {**block, **census.data[geoid]}
        merged.append(entry)

print(f"Blocks with demographics: {len(merged)}")
print(f"Sample keys: {list(merged[0].keys())}")

## 9. Aggregate Statistics

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"geoid": geoid, **data}
    for geoid, data in census.data.items()
])

print("=== Denver 15-min Drive Area ===")
print(f"Block groups:       {len(df)}")
print(f"Total population:   {df['population'].sum():,.0f}")
print(f"Avg population/BG:  {df['population'].mean():,.0f}")
print(f"Median income range: ${df['median_income'].min():,.0f} – ${df['median_income'].max():,.0f}")
print(f"Mean median income:  ${df['median_income'].mean():,.0f}")

## Summary

| What you learned | API |
|---|---|
| Get census data from an isochrone | `get_census_data(iso, ["population"])` |
| Use friendly names or B-codes | `"median_income"` or `"B19013_001E"` |
| Query by GEOID list | `get_census_data(["060750201001", ...], variables)` |
| Query by point | `get_census_data((lat, lon), variables)` |
| Merge with geometry for maps | Loop over blocks, merge by GEOID |

**Next notebook:** [04 — Choropleth Maps](04-choropleth-maps.ipynb)